# **Circuits: activation patching on the IOI task**

The IOI task (indirect object identification): in the sentence «When John and Mary went to the shops, Mary gave the bag to» the model should continue with « John», the name that was not repeated. [Wang et al., 2022](https://arxiv.org/abs/2211.00593) found a circuit of 26 attention heads in GPT-2 small that solves this task. In this practice you repeat the main technique of their work, activation patching, and see in which layers and heads the answer lives.

The tasks are numbered; you enter the answers on the course website, in the lesson «Circuits: Practice».

## **Set up**
A CPU is enough; a GPU makes the run faster.

In [ ]:
!pip install transformer-lens==3.9.0 -q

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)  # only forward passes here
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = HookedTransformer.from_pretrained("gpt2-small", device=device)

## **Data: clean and corrupted prompts**

Each clean prompt has two names: IO (the indirect object, the correct answer) and S (the subject, mentioned twice). In the corrupted prompt the second mention of S is replaced with IO, so the model no longer has a correct answer. The pairs differ in exactly one token, so patching shows where the model keeps exactly this difference.

In [ ]:
names = [(" John", " Mary"), (" Tom", " James"), (" Dan", " Sid"), (" Martin", " Amy")]
places = [("shops", "bag"), ("park", "ball")]

clean, corrupted, io_names, s_names = [], [], [], []
for a, b in names:
    for place, obj in places:
        for first, second in [(a, b), (b, a)]:
            # clean prompt: the second mention (S2) is the second name, the answer (IO) is the first
            clean.append(f"When{first} and{second} went to the {place},{second} gave the {obj} to")
            # corrupted: S2 is replaced with the other name, the rest is the same
            corrupted.append(f"When{first} and{second} went to the {place},{first} gave the {obj} to")
            io_names.append(first)
            s_names.append(second)

clean_tokens = model.to_tokens(clean)
corrupted_tokens = model.to_tokens(corrupted)
io_ids = torch.tensor([model.to_single_token(n) for n in io_names], device=device)
s_ids = torch.tensor([model.to_single_token(n) for n in s_names], device=device)

print(len(clean), "prompt pairs, length", clean_tokens.shape[1], "tokens")
print(clean[0])
print(corrupted[0])
print(model.to_str_tokens(clean_tokens[0]))

## **Metric: logit difference**

We measure $\mathrm{logit}(IO) - \mathrm{logit}(S)$ at the last position, averaged over 16 prompts. A positive value means the model prefers the correct name.

In [ ]:
def logit_diff(logits):
    """Mean over prompts of the IO-minus-S logit difference at the last position."""
    last = logits[:, -1]
    return ...  # Your code here


clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits = model(corrupted_tokens)
clean_ld, corrupted_ld = logit_diff(clean_logits).item(), logit_diff(corrupted_logits).item()
print(f"clean prompts: {clean_ld:.2f}, corrupted: {corrupted_ld:.2f}")

**Task 1. What is the mean logit difference on the clean prompts? Round the answer to two decimal places.**

**Task 2. What is it on the corrupted prompts? Round the answer to two decimal places.**

## **Patching the residual stream**

We run the corrupted prompts and, at one layer and one position, substitute the residual stream from the clean run. If the logit difference comes back, the information needed was stored at that place. We normalise the result: 0 as on the corrupted prompt, 1 as on the clean one.

In [ ]:
def recovered(logits):
    """The fraction of the logit difference restored by the patch: 0 as on the corrupted prompt, 1 as on the clean one."""
    return (logit_diff(logits).item() - corrupted_ld) / (clean_ld - corrupted_ld)


n_layers, n_pos = model.cfg.n_layers, clean_tokens.shape[1]
resid = torch.zeros(n_layers, n_pos)
for layer in range(n_layers):
    for pos in range(n_pos):
        def patch(act, hook, pos=pos):
            act[:, pos] = clean_cache[hook.name][:, pos]  # the clean activation at one position
            return act
        out = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(utils.get_act_name("resid_pre", layer), patch)])
        resid[layer, pos] = recovered(out)

labels = [f"{t}_{i}" for i, t in enumerate(model.to_str_tokens(clean_tokens[0]))]
plt.figure(figsize=(10, 4))
plt.imshow(resid, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(n_pos), labels, rotation=90)
plt.ylabel("layer (resid_pre)")
plt.colorbar(label="recovered logit difference")
plt.show()

s2_pos, end_pos = 10, n_pos - 1
print("position S2:", [round(v, 2) for v in resid[:, s2_pos].tolist()])
print("last position:", [round(v, 2) for v in resid[:, end_pos].tolist()])

**Task 3. From which layer on does patching the residual stream at the last position restore more than half of the logit difference?**

**Task 4. Where is the information needed stored in the layers before that?**

## **Patching attention heads**

Now we substitute the output of one attention head (`hook_z`) from the clean run at all positions — 12 layers of 12 heads. A head with a large positive effect carries the information needed; a head with a negative effect works against the answer.

In [ ]:
n_heads = model.cfg.n_heads
heads = torch.zeros(n_layers, n_heads)
for layer in range(n_layers):
    for head in range(n_heads):
        def patch(z, hook, head=head):
            z[:, :, head] = clean_cache[hook.name][:, :, head]  # the clean output of one head at all positions
            return z
        out = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(utils.get_act_name("z", layer), patch)])
        heads[layer, head] = recovered(out)

plt.figure(figsize=(6, 5))
plt.imshow(heads, cmap="RdBu", vmin=-0.6, vmax=0.6)
plt.xlabel("head")
plt.ylabel("layer")
plt.colorbar(label="recovered logit difference")
plt.show()

# Your code here: print the five heads with the largest effect and the head with the smallest (most negative) one

**Task 5. What is the effect of the head with the most negative effect? Round the answer to two decimal places.**

**Task 6. Which head is it?** Wang et al. call such heads negative name movers: they write into the residual stream against the name they attend to.

## **Conclusion**

- The information that tells the clean prompt from the corrupted one first lives at the position of the second mention of S and moves to the last position in the middle layers: patching the residual stream shows this movement layer by layer.
- Head patching singles out a few heads out of 144; their roles match those found by Wang et al.: heads that detect the repeated name, S-inhibition heads in layers 7–8, name movers in layers 9–10, and negative name movers that work against the answer.
- Patching shows where the information needed for this difference on these prompts is stored. A circuit drawn from it is a hypothesis to be checked on other prompt templates, another metric and another way of corrupting the input.